In [1]:
import numpy as np, pandas as pd
rng = np.random.default_rng(42)

n_users, n_days = 150, 90
feats = ["logons", "offhours_logons", "usb_events", "files_copied", "email_attachments", "distinct_pcs"]

# each user gets personal baseline rates
base = pd.DataFrame({
    "logons": rng.uniform(2, 6, n_users),
    "offhours_logons": rng.uniform(0.0, 0.5, n_users),
    "usb_events": rng.uniform(0.0, 1.0, n_users),
    "files_copied": rng.lognormal(2.0, 0.8, n_users),     # a few users copy a lot, on purpose
    "email_attachments": rng.uniform(0.5, 4, n_users),
    "distinct_pcs": rng.uniform(1.0, 2.0, n_users),
})

rows = []
insiders = rng.choice(n_users, 8, replace=False)
start_bad = rng.integers(65, 80, size=8)                  # each insider turns bad at a different day
bad_start = dict(zip(insiders, start_bad))

for u in range(n_users):
    for d in range(n_days):
        r = {f: rng.poisson(base.loc[u, f]) for f in feats}
        r.update(user=u, day=d, label=0)
        if u in bad_start and d >= bad_start[u]:           # insider behavior: after-hours, USB, mass copying
            r["offhours_logons"] += rng.poisson(3)
            r["usb_events"] += rng.poisson(4)
            r["files_copied"] += rng.poisson(base.loc[u, "files_copied"] * 3 + 20)
            r["distinct_pcs"] += rng.poisson(2)
            r["label"] = 1
        rows.append(r)

df = pd.DataFrame(rows).sort_values(["user", "day"]).reset_index(drop=True)
print(df.shape, "| malicious user-days:", df["label"].sum(), "| insiders:", len(insiders))

(13500, 9) | malicious user-days: 120 | insiders: 8


In [2]:
g = df.groupby("user")[feats]
mean = g.transform(lambda s: s.shift(1).rolling(30, min_periods=10).mean())
std = g.transform(lambda s: s.shift(1).rolling(30, min_periods=10).std())
z = (df[feats] - mean) / std.clip(lower=1.0)              # clip avoids dividing by tiny std
z = z.add_suffix("_z").fillna(0)
data = pd.concat([df, z], axis=1)
z_cols = list(z.columns)
print(data[data["label"] == 1][z_cols].mean().round(1))    # attack days should show large z
print(data[data["label"] == 0][z_cols].mean().round(1))    # normal days should sit near 0

logons_z               0.1
offhours_logons_z      1.7
usb_events_z           1.8
files_copied_z         3.0
email_attachments_z   -0.0
distinct_pcs_z         1.0
dtype: float64
logons_z               0.0
offhours_logons_z     -0.0
usb_events_z           0.0
files_copied_z         0.0
email_attachments_z    0.0
distinct_pcs_z         0.0
dtype: float64


In [3]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

train = data[data["day"] < 60]
test = data[data["day"] >= 60].copy()

def run(cols, name):
    iso = IsolationForest(n_estimators=200, random_state=42, n_jobs=-1).fit(train[cols])
    test[name] = -iso.score_samples(test[cols])            # higher = more abnormal
    print(f"{name}: PR-AUC = {average_precision_score(test['label'], test[name]):.4f}")

run(feats, "score_raw")
run(z_cols, "score_peruser")
print("baseline (random):", round(test["label"].mean(), 4))

score_raw: PR-AUC = 0.9829
score_peruser: PR-AUC = 0.5768
baseline (random): 0.0267


In [4]:
def recall_at_k(col, frac):
    k = int(len(test) * frac)
    top = test.nlargest(k, col)
    return top["label"].sum() / test["label"].sum()

for col in ["score_raw", "score_peruser"]:
    print(col, {f"top {int(f*100)}%": round(recall_at_k(col, f), 3) for f in [0.01, 0.05]})

# did we catch every insider at least once?
top5 = test.nlargest(int(len(test) * 0.05), "score_peruser")
print("insiders flagged:", top5.loc[top5["label"] == 1, "user"].nunique(), "of", len(insiders))

score_raw {'top 1%': np.float64(0.375), 'top 5%': np.float64(0.983)}
score_peruser {'top 1%': np.float64(0.333), 'top 5%': np.float64(0.658)}
insiders flagged: 8 of 8


In [5]:
import os
os.makedirs("../data/processed", exist_ok=True)
test[["user", "day", "label", "score_peruser"]].to_csv("../data/processed/insider_scores.csv", index=False)